# Generate PSTH zarr files

Runs `extract_neuron_psth_to_zarr` (0.2s bins) for a list of sessions and saves the results to `/root/capsule/scratch/psth_results/` as `{session}_0.2s.zarr`, which is the format expected by the raster / scatter workers.

Code adapted from `psth_playground.ipynb`.

In [1]:
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

In [2]:
from joblib import Parallel, delayed
import os, traceback, pandas as pd
from nwb_utils import NWBUtils
from create_psth import extract_neuron_psth_to_zarr

SAVE_DIR = "/root/capsule/scratch/psth_results/"
ALIGN_TO = ["go_cue", "reward_go_cue_start", "trial_start"]
TIME_WINDOW = (-6, 6)
BIN_SIZE = 0.2  # produces the "_0.2s.zarr" suffix expected by the raster worker

sessions = [
    "ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14",
    "ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17",
    "ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58",
    "ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38",
    "ecephys_839483_2026-05-26_16-02-39_sorted_2026-06-09_09-42-36",
    "ecephys_839483_2026-05-28_14-47-06_sorted-bandpass_2026-07-17_20-18-46",
]

os.makedirs(SAVE_DIR, exist_ok=True)


def worker(session_name):
    """Return a status dict; never raise so Parallel keeps running."""
    try:
        nwb_data, _ = NWBUtils.combine_nwb(session_name=session_name)
        save_name = f"{session_name}_{BIN_SIZE}s"
        extract_neuron_psth_to_zarr(
            nwb_data=nwb_data,
            align_to_event=ALIGN_TO,
            time_window=TIME_WINDOW,
            bin_size=BIN_SIZE,
            save_folder=SAVE_DIR,
            save_name=save_name,
        )
        return {"session": session_name, "ok": True, "error": "", "message": ""}
    except Exception as e:
        return {
            "session": session_name,
            "ok": False,
            "error": type(e).__name__,
            "message": str(e),
            "traceback": "".join(traceback.format_exception_only(type(e), e)).strip(),
        }


n_jobs = max(1, (int(os.getenv("CO_CPUS", os.cpu_count() or 2)) - 1))
results = Parallel(n_jobs=n_jobs, backend="loky")(delayed(worker)(s) for s in sessions)

ok = [r for r in results if r["ok"]]
fail = [r for r in results if not r["ok"]]
print(f"Succeeded: {len(ok)} / {len(results)}")
for r in fail:
    print(f"  FAILED {r['session']}: {r['error']} - {r['message']}")

Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-05_14-36-28_sorted-bandpass_2026-07-18_00-37-38/nwb/ecephys_839480_2026-06-05_14-36-28_experiment1_recording1.nwb
Found ephys NWB: /root/capsule/data/ecephys_839483_2026-05-28_14-47-06_sorted-bandpass_2026-07-17_20-18-46/nwb/ecephys_839483_2026-05-28_14-47-06_experiment1_recording1.nwb
Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14/nwb/ecephys_839480_2026-06-02_16-20-58_experiment1_recording1.nwb
Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-04_13-45-44_sorted-bandpass_2026-07-17_22-26-58/nwb/ecephys_839480_2026-06-04_13-45-44_experiment1_recording1.nwb
Found ephys NWB: /root/capsule/data/ecephys_839483_2026-05-26_16-02-39_sorted_2026-06-09_09-42-36/nwb/ecephys_839483_2026-05-26_16-02-39_experiment1_recording1.nwb
Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17/nwb/ecephys_839480_2026-06-03_15-09-14_experim